In [6]:
import numpy as np
import pandas as pd
import sys
import os
from rdkit import Chem
sys.path.append(os.path.abspath(os.path.join('..')))
sys.modules.pop("functionality.data_preparation", None)
from functionality.data_preparation import FingerprintDataset
from torch.utils.data import DataLoader
from lightgbm import LGBMClassifier
import torch 
import os
from tqdm import tqdm
from fastparquet import write
import pyarrow as pa
import pyarrow.parquet as pq
import joblib

In [16]:
def train_model_for_protein(protein_data, protein_name, batch_size=1000, lgb_params=None):
    """
    Train a LightGBM model for a protein using PyTorch DataLoader batching

    Args:
        protein_data (pd.DataFrame): Data containing SMILES strings and labels
        protein_name (str): Name of the protein
        batch_size (int): Number of samples per batch
        lgb_params (dict): Parameters for the LightGBM classifier

    Returns:
        LGBMClassifier: Trained LightGBM model with the best iteration
    """
    # Initialize Dataset & DataLoader
    dataset = FingerprintDataset(protein_data['molecule_smiles'].tolist(), protein_data['binds'].tolist())
    def custom_collate_fn(batch):
        fingerprints = [item[0] for item in batch]
        bit_info = [item[1] for item in batch] if batch[0][1] is not None else None
        labels = [item[2] for item in batch]
    
        fingerprints = torch.stack(fingerprints)
        labels = torch.stack(labels)
    
        if bit_info is not None:
            return fingerprints, bit_info, labels
        else:
            return fingerprints, labels
        
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=4,
        shuffle=False,
        collate_fn=custom_collate_fn 
    )

    # Initialize LightGBM Classifier
    lgb_cls = LGBMClassifier(**lgb_params)

    # Train model in batches
    for X_batch, y_batch in tqdm(dataloader, desc=f"Training {protein_name}"):
        X_batch = X_batch.numpy()
        y_batch = y_batch.numpy()

        # Convert to Pandas DataFrame
        num_features = X_batch.shape[1]
        df_X_batch = pd.DataFrame(X_batch, columns=[f"feature_{j}" for j in range(num_features)])
        df_y_batch = pd.DataFrame(y_batch, columns=["label"])

        # Train LightGBM
        lgb_cls.fit(
            df_X_batch, 
            df_y_batch.values.ravel(), 
            eval_metric='auc', 
            init_model=lgb_cls.booster_ if hasattr(lgb_cls, "booster_") else None
        )

    # Get the best iteration for early stopping
    best_iteration = lgb_cls.best_iteration_
    print(f"Best iteration for {protein_name}: {best_iteration}")

    # Set the final model to the best iteration
    lgb_cls.set_params(n_estimators=best_iteration)

    return lgb_cls


def save_models(model, protein, save_dir="../checkpoints"):
    """
    Save trained LightGBM model to disk

    Args:
        model (LGBMClassifier): Trained LightGBM model
        protein (str): Name of the protein
        save_dir (str): Directory to save the model
    """
    os.makedirs(save_dir, exist_ok=True)
    model_path = os.path.join(save_dir, f"{protein}_lightgbm.pkl")

    joblib.dump(model, model_path)
    print(f"LightGBM model for {protein} saved at {model_path}")

def train_models_by_protein(batch_size=100000, save_dir="../checkpoints", lgb_params=None):
    """
    Train models for multiple proteins and save them

    Args:
        batch_size (int): Number of samples per batch
        save_dir (str): Directory to save models
        lgb_params (dict): LightGBM parameters

    Returns:
        str: Confirmation message
    """
    protein_names = ['sEH', 'BRD4', 'HSA']

    for protein in tqdm(protein_names, desc="Training models"):
        train_data = pd.read_parquet(f'../intermediates/train_data/{protein}/{protein}_train.parquet')
        val_data = pd.read_parquet(f'../intermediates/train_data/{protein}/{protein}_val.parquet')

        model = train_model_for_protein(train_data, protein, batch_size, lgb_params)
        save_models(model, protein, save_dir)

    return "Training complete"

In [18]:
import lightgbm as lgb
print(lgb.__version__)

4.5.0


In [8]:
def save_eval_set(eval_set, protein_name, batch_size=1000, save_path="../intermediates/embeddings/"):
    """Efficiently saves evaluation data in batches"""
    
    if eval_set is None or eval_set.empty:
        return None
    
    save_path = os.path.join(save_path, f"{protein_name}_lightgbm_val.parquet")

    # Create the dataset (set bit_info=False for evaluation)
    dataset = FingerprintDataset(eval_set['molecule_smiles'], eval_set['binds'])
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=0, shuffle=False)
    
    # Define the Parquet schema
    PARQUET_SCHEMA = pa.schema(
        [pa.field(f"fp_{i}", pa.int8()) for i in range(dataset.fp_size)] +  
        [pa.field("label", pa.int64())]  
    )

    # Initialize Parquet writer
    with pq.ParquetWriter(save_path, PARQUET_SCHEMA, compression="SNAPPY") as writer:
        for batch_fingerprints, batch_labels in tqdm(dataloader, desc="Computing Fingerprints"):
            # Convert batch tensors to NumPy arrays
            batch_fingerprints = batch_fingerprints.numpy()
            batch_labels = batch_labels.numpy()

            # Create a DataFrame for the batch
            df_batch = pd.DataFrame(
                batch_fingerprints,
                columns=[f"fp_{i}" for i in range(dataset.fp_size)]
            )
            df_batch["label"] = batch_labels

            # Convert the DataFrame to an Arrow Table
            table = pa.Table.from_pandas(df_batch, schema=PARQUET_SCHEMA)
            writer.write_table(table)

    print(f"Evaluation set saved to {save_path}")

In [17]:
lgb_params = {
        'max_depth': 11,
        'bagging_fraction': 0.9,
        'learning_rate': 0.05,
        'colsample_bytree': 1,
        'colsample_bynode': 0.5,
        'lambda_l1': 1,
        'objective': 'binary',
        'lambda_l2': 1.5,
        'num_leaves': 490,
        'min_data_in_leaf': 50,
        'verbose': -1,
        'metric': 'average_precision',
        'device': 'cpu'
    }

train_models_by_protein(lgb_params=lgb_params)

Training sEH: 100%|██████████| 37/37 [15:40<00:00, 25.42s/it]


Best iteration for sEH: 0


Training models:  33%|███▎      | 1/3 [15:43<31:26, 943.31s/it]

LightGBM model for sEH saved at ../checkpoints/sEH_lightgbm.pkl


Training BRD4: 100%|██████████| 35/35 [13:55<00:00, 23.88s/it]


...

Training HSA: 100%|██████████| 34/34 [14:50<00:00, 26.18s/it]


Best iteration for HSA: 0


Training models: 100%|██████████| 3/3 [44:34<00:00, 891.60s/it]

LightGBM model for HSA saved at ../checkpoints/HSA_lightgbm.pkl


'Training complete'

In [38]:
protein_names = ['sEH', 'BRD4', 'HSA']

for protein in protein_names:

    val_data = pd.read_parquet(f'../intermediates/train_data/{protein}/{protein}_val.parquet')
    save_eval_set(val_data, protein)

Computing Fingerprints:   4%|▍         | 16/406 [01:22<06:23,  1.02it/s] Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72bc1243f370>
Traceback (most recent call last):
  File "/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72bc1243f370>
Traceback (most recent call last):
  File "/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    s

Evaluation set saved to ../intermediates/embeddings/sEH_lightgbm_val.parquet


Computing Fingerprints: 100%|██████████| 380/380 [00:49<00:00,  7.62it/s]


Evaluation set saved to ../intermediates/embeddings/BRD4_lightgbm_val.parquet


Computing Fingerprints: 100%|██████████| 375/375 [00:51<00:00,  7.32it/s]


Evaluation set saved to ../intermediates/embeddings/HSA_lightgbm_val.parquet
